# Credit Card Fraud: Time-Aware Model Selection
The latest 30% of training transactions form the validation period. Models are ranked by **PR-AUC (average precision)** because fraud is rare. The external `fraud_test.csv` is not loaded or evaluated in this notebook.

In [ ]:
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score, RocCurveDisplay,
    PrecisionRecallDisplay,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

pd.set_option('display.max_columns', 50)
plt.style.use('seaborn-v0_8-whitegrid')

RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd()
TRAIN_PATH = PROJECT_ROOT / 'Data' / 'raw' / 'fraud_train.csv'
RESULTS_PATH = PROJECT_ROOT / 'Data' / 'processed' / 'time_aware_model_comparison.csv'

if not TRAIN_PATH.exists():
    raise FileNotFoundError(f'Missing official training data: {TRAIN_PATH}')

## Loading Data
The same feature will be use for every model, and the features that are selected are as a result of the EDA process in 01 Notebook

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorized great-circle distance between two coordinate pairs."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    delta_lat = lat2 - lat1
    delta_lon = lon2 - lon1
    a = (
        np.sin(delta_lat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(delta_lon / 2) ** 2
    )
    return 2 * 6371.0088 * np.arcsin(np.sqrt(a))


def prepare_model_features(raw):
    data = raw.drop(
        columns=[c for c in raw.columns if c.startswith('Unnamed:')],
        errors='ignore',
    ).copy()
    transaction_time = pd.to_datetime(
        data['trans_date_trans_time'], format='mixed', dayfirst=True, errors='coerce'
    )
    birth_date = pd.to_datetime(
        data['dob'], format='mixed', dayfirst=True, errors='coerce'
    )

    features = pd.DataFrame(index=data.index)
    features['amt_log1p'] = np.log1p(data['amt'])
    features['city_pop_log1p'] = np.log1p(data['city_pop'])
    features['age_at_transaction'] = (transaction_time - birth_date).dt.days / 365.2425
    features['trans_hour'] = transaction_time.dt.hour
    features['trans_day_of_week'] = transaction_time.dt.dayofweek
    features['trans_month'] = transaction_time.dt.month
    features['is_weekend'] = transaction_time.dt.dayofweek.ge(5).astype('int8')
    features['is_night'] = transaction_time.dt.hour.between(0, 5).astype('int8')
    features['hour_sin'] = np.sin(2 * np.pi * transaction_time.dt.hour / 24)
    features['hour_cos'] = np.cos(2 * np.pi * transaction_time.dt.hour / 24)
    features['day_sin'] = np.sin(2 * np.pi * transaction_time.dt.dayofweek / 7)
    features['day_cos'] = np.cos(2 * np.pi * transaction_time.dt.dayofweek / 7)
    features['merchant_distance_km'] = haversine_km(
        data['lat'].to_numpy(), data['long'].to_numpy(),
        data['merch_lat'].to_numpy(), data['merch_long'].to_numpy(),
    )
    features['lat'] = data['lat']
    features['long'] = data['long']

    for column in ['category', 'gender', 'state']:
        features[column] = data[column].astype('string').str.strip().fillna('missing')

    return features, data['is_fraud'].astype('int8'), transaction_time


identifier_types = {'cc_num': 'string', 'zip': 'string', 'trans_num': 'string'}
official_train_raw = pd.read_csv(TRAIN_PATH, dtype=identifier_types, low_memory=False)

if len(official_train_raw) != 1_296_675:
    raise ValueError(
        f'Expected 1,296,675 official training rows, found {len(official_train_raw):,}.'
    )

X_all, y_all, transaction_time = prepare_model_features(official_train_raw)
print(f'Official training rows: {len(X_all):,}')
print(f'Fraud cases: {y_all.sum():,} ({y_all.mean():.3%})')

### 70/30 Training and Validation Split
Again, we will split the data using the transaction time

In [ ]:
order = np.argsort(transaction_time.to_numpy(), kind='stable')
X_all = X_all.iloc[order].reset_index(drop=True)
y_all = y_all.iloc[order].reset_index(drop=True)
transaction_time = transaction_time.iloc[order].reset_index(drop=True)

split_index = int(len(X_all) * 0.70)
X_train, X_validation = X_all.iloc[:split_index], X_all.iloc[split_index:]
y_train, y_validation = y_all.iloc[:split_index], y_all.iloc[split_index:]
time_train, time_validation = transaction_time.iloc[:split_index], transaction_time.iloc[split_index:]

split_summary = pd.DataFrame({
    'partition': ['Earliest 70%: development', 'Latest 30%: validation'],
    'rows': [len(X_train), len(X_validation)],
    'start': [time_train.min(), time_validation.min()],
    'end': [time_train.max(), time_validation.max()],
    'fraud_cases': [int(y_train.sum()), int(y_validation.sum())],
    'fraud_rate': [y_train.mean(), y_validation.mean()],
})
display(split_summary)

ax = split_summary.set_index('partition')['fraud_rate'].mul(100).plot.bar(
    color=['#4C78A8', '#F58518'], figsize=(8, 4)
)
ax.set_title('Fraud Rate by Chronological Partition')
ax.set_xlabel('')
ax.set_ylabel('Fraud rate (%)')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

### Preliminary Model Training
- Since this dataset is extremely imbalance, we have deliberately engineered. First notice the imbalance ratio is roughly 172
- The goal at this moment is to determine which model family naturally handled fraud detection best before investing time in hyperparameter tuning

In [ ]:
numeric_model_features = [
    'amt_log1p', 'city_pop_log1p', 'age_at_transaction', 'trans_hour',
    'trans_day_of_week', 'trans_month', 'is_weekend', 'is_night',
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos',
    'merchant_distance_km', 'lat', 'long',
]
categorical_model_features = ['category', 'gender', 'state']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one_hot', OneHotEncoder(handle_unknown='ignore', min_frequency=20)),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_transformer, numeric_model_features),
    ('categorical', categorical_transformer, categorical_model_features),
])

negative_count = int(y_train.eq(0).sum())
positive_count = int(y_train.eq(1).sum())
scale_pos_weight = negative_count / positive_count
print(f'Training negative/positive class weight: {scale_pos_weight:.1f}')

pipeline_estimators = {
    'LogisticRegression': LogisticRegression(
        class_weight='balanced', solver='saga', max_iter=400, tol=2e-3,
        random_state=RANDOM_STATE,
    ),
    'DecisionTreeClassifier': DecisionTreeClassifier(
        class_weight='balanced', max_depth=14, min_samples_leaf=20,
        random_state=RANDOM_STATE,
    ),
    'RandomForestClassifier': RandomForestClassifier(
        n_estimators=160, class_weight='balanced_subsample', max_depth=18,
        min_samples_leaf=5, max_features='sqrt', n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    'ExtraTreesClassifier': ExtraTreesClassifier(
        n_estimators=160, class_weight='balanced', max_depth=18,
        min_samples_leaf=5, max_features='sqrt', n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    'LightGBMClassifier': LGBMClassifier(
        n_estimators=350, learning_rate=0.06, num_leaves=31,
        class_weight='balanced', subsample=0.9, colsample_bytree=0.9,
        n_jobs=-1, random_state=RANDOM_STATE, verbosity=-1,
    ),
    'XGBoostClassifier': XGBClassifier(
        n_estimators=350, learning_rate=0.06, max_depth=7,
        min_child_weight=3, subsample=0.9, colsample_bytree=0.9,
        scale_pos_weight=scale_pos_weight, objective='binary:logistic',
        eval_metric='aucpr', tree_method='hist', n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
}

pipelines = {
    name: Pipeline([('preprocessor', clone(preprocessor)), ('model', estimator)])
    for name, estimator in pipeline_estimators.items()
}

catboost_model = CatBoostClassifier(
    iterations=350, depth=8, learning_rate=0.06,
    loss_function='Logloss', eval_metric='PRAUC:type=Classic',
    auto_class_weights='Balanced', random_seed=RANDOM_STATE,
    verbose=False, allow_writing_files=False,
)

In [ ]:
def classification_metrics(model_name, model, X_fit, y_fit, X_eval, y_eval):
    train_prediction = model.predict(X_fit).astype('int8').ravel()
    prediction = model.predict(X_eval).astype('int8').ravel()
    probability = model.predict_proba(X_eval)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_eval, prediction).ravel()
    return {
        'Model Name': model_name,
        'Train Accuracy': accuracy_score(y_fit, train_prediction),
        'Validation Accuracy': accuracy_score(y_eval, prediction),
        'Precision': precision_score(y_eval, prediction, zero_division=0),
        'Recall': recall_score(y_eval, prediction, zero_division=0),
        'F1 Score': f1_score(y_eval, prediction, zero_division=0),
        'ROC-AUC': roc_auc_score(y_eval, probability),
        'PR-AUC': average_precision_score(y_eval, probability),
        'False Positives': int(fp),
        'False Negatives': int(fn),
    }


benchmark_rows = []
fitted_models = {}

for name, pipeline in pipelines.items():
    started = perf_counter()
    pipeline.fit(X_train, y_train)
    fitted_models[name] = pipeline
    row = classification_metrics(
        name, pipeline, X_train, y_train, X_validation, y_validation
    )
    row['Fit Seconds'] = perf_counter() - started
    benchmark_rows.append(row)
    print(f'Finished {name} in {row["Fit Seconds"]:.1f} seconds')

started = perf_counter()
catboost_model.fit(
    X_train, y_train,
    cat_features=categorical_model_features,
    eval_set=(X_validation, y_validation),
    early_stopping_rounds=40,
    use_best_model=True,
)
fitted_models['CatBoostClassifier'] = catboost_model
catboost_row = classification_metrics(
    'CatBoostClassifier', catboost_model,
    X_train, y_train, X_validation, y_validation,
)
catboost_row['Fit Seconds'] = perf_counter() - started
benchmark_rows.append(catboost_row)
print(f'Finished CatBoostClassifier in {catboost_row["Fit Seconds"]:.1f} seconds')

## 4. PR-AUC comparison

PR-AUC is the primary ranking measure. Accuracy is included only for completeness and must not drive selection in this highly imbalanced dataset. Precision, recall, F1, ROC-AUC, and error counts describe the operational trade-offs at each model's default threshold.
Notice that we will not judge this model using accuracy since this dataset is imbalance, instead, we will be using PR-AUC, since the goal is to detect as many fraud transactions as possible (high recall) and minimizing unnecessary investigations of legitimate transactions (precision)

#### Observations
- Gradient boosting models dominate this fraud-detection problem, lightGMB achieves the highest overall ranking performance.

In [ ]:
benchmark_results = (
    pd.DataFrame(benchmark_rows)
      .sort_values(['PR-AUC', 'Recall'], ascending=False)
      .reset_index(drop=True)
)

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
benchmark_results.to_csv(RESULTS_PATH, index=False)

display(benchmark_results.style.format({
    'Train Accuracy': '{:.4f}', 'Validation Accuracy': '{:.4f}',
    'Precision': '{:.4f}', 'Recall': '{:.4f}', 'F1 Score': '{:.4f}',
    'ROC-AUC': '{:.4f}', 'PR-AUC': '{:.4f}', 'Fit Seconds': '{:.1f}',
}).highlight_max(subset=['F1 Score', 'ROC-AUC', 'PR-AUC'], color='#C6EFCE'))

ax = benchmark_results.sort_values('PR-AUC').plot.barh(
    x='Model Name', y='PR-AUC', figsize=(9, 5), legend=False, color='#E45756'
)
ax.axvline(y_validation.mean(), color='gray', linestyle='--', label='Fraud-rate baseline')
ax.set_title('Time-Aware Validation: PR-AUC by Model')
ax.set_xlabel('PR-AUC (average precision)')
ax.set_ylabel('')
ax.legend()
plt.tight_layout()
plt.show()

best_model_name = benchmark_results.iloc[0]['Model Name']
print(f'Selected model family by chronological validation PR-AUC: {best_model_name}')
print(f'Results saved to: {RESULTS_PATH}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for name, model in fitted_models.items():
    probability = model.predict_proba(X_validation)[:, 1]
    RocCurveDisplay.from_predictions(y_validation, probability, name=name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_validation, probability, name=name, ax=axes[1])
axes[0].set_title('ROC Curves: Chronological Validation')
axes[1].set_title('Precision–Recall Curves: Chronological Validation')
plt.show()

**What this block does:** Draws ROC and precision–recall curves for all preliminary models on the same chronological validation data. These curves compare ranking quality across possible decision thresholds rather than at only the default cutoff.

## 5. Time-aware tuning: LightGBM and XGBoost

Essentially, this part is splitting the data into three parts: training set to fit the model, the validation set to tune the hyperparameters and the testing set for the final evaluation. Specifically, we have decided to use the last 300k in the original training set to do the validation, and then use the tune result to retrain the training set. This would avoid data leakage

In [ ]:
from itertools import product

from lightgbm import early_stopping as lgb_early_stopping
from lightgbm import log_evaluation as lgb_log_evaluation

# Use the most recent 300,000 rows inside development for computationally
# responsible screening while preserving chronological order.
tuning_window_size = min(300_000, len(X_train))
X_tuning_window = X_train.iloc[-tuning_window_size:]
y_tuning_window = y_train.iloc[-tuning_window_size:]
inner_split_index = int(len(X_tuning_window) * 0.70)
X_inner_fit = X_tuning_window.iloc[:inner_split_index]
X_inner_tune = X_tuning_window.iloc[inner_split_index:]
y_inner_fit = y_tuning_window.iloc[:inner_split_index]
y_inner_tune = y_tuning_window.iloc[inner_split_index:]

# Fit preprocessing only on the earlier inner-fit period.
tuning_preprocessor = clone(preprocessor)
X_inner_fit_encoded = tuning_preprocessor.fit_transform(X_inner_fit)
X_inner_tune_encoded = tuning_preprocessor.transform(X_inner_tune)

inner_summary = pd.DataFrame({
    'partition': ['Inner fit', 'Inner tuning', 'Locked final validation'],
    'rows': [len(X_inner_fit), len(X_inner_tune), len(X_validation)],
    'fraud_cases': [int(y_inner_fit.sum()), int(y_inner_tune.sum()), int(y_validation.sum())],
    'fraud_rate': [y_inner_fit.mean(), y_inner_tune.mean(), y_validation.mean()],
})
display(inner_summary)

inner_scale_pos_weight = y_inner_fit.eq(0).sum() / y_inner_fit.eq(1).sum()
print(f'Inner-fit negative/positive weight: {inner_scale_pos_weight:.2f}')

In [ ]:
lightgbm_candidates = [
    {'num_leaves': 15, 'max_depth': -1, 'min_child_samples': 20,
     'reg_alpha': 0.0, 'reg_lambda': 0.0},
    {'num_leaves': 31, 'max_depth': -1, 'min_child_samples': 20,
     'reg_alpha': 0.0, 'reg_lambda': 0.0},
    {'num_leaves': 31, 'max_depth': -1, 'min_child_samples': 50,
     'reg_alpha': 0.0, 'reg_lambda': 1.0},
    {'num_leaves': 63, 'max_depth': -1, 'min_child_samples': 50,
     'reg_alpha': 0.0, 'reg_lambda': 2.0},
]

lightgbm_tuning_rows = []
for candidate_number, params in enumerate(lightgbm_candidates, start=1):
    started = perf_counter()
    candidate = LGBMClassifier(
        n_estimators=500, learning_rate=0.05,
        class_weight='balanced', subsample=0.9, colsample_bytree=0.9,
        n_jobs=-1, random_state=RANDOM_STATE, verbosity=-1,
        **params,
    )
    candidate.fit(
        X_inner_fit_encoded, y_inner_fit,
        eval_set=[(X_inner_tune_encoded, y_inner_tune)],
        eval_metric='average_precision',
        callbacks=[lgb_early_stopping(40, verbose=False), lgb_log_evaluation(0)],
    )
    tune_probability = candidate.predict_proba(X_inner_tune_encoded)[:, 1]
    lightgbm_tuning_rows.append({
        'candidate': candidate_number,
        'PR-AUC': average_precision_score(y_inner_tune, tune_probability),
        'best_iteration': int(candidate.best_iteration_),
        'fit_seconds': perf_counter() - started,
        **params,
    })
    print(f'LightGBM candidate {candidate_number}/{len(lightgbm_candidates)} complete')

lightgbm_tuning_results = (
    pd.DataFrame(lightgbm_tuning_rows)
      .sort_values('PR-AUC', ascending=False)
      .reset_index(drop=True)
)
display(lightgbm_tuning_results.style.format({
    'PR-AUC': '{:.4f}', 'fit_seconds': '{:.1f}',
}))

best_lightgbm_row = lightgbm_tuning_results.iloc[0]
best_lightgbm_params = {
    key: (int(best_lightgbm_row[key]) if key in {'num_leaves', 'max_depth', 'min_child_samples'}
          else float(best_lightgbm_row[key]))
    for key in ['num_leaves', 'max_depth', 'min_child_samples', 'reg_alpha', 'reg_lambda']
}
best_lightgbm_iterations = int(best_lightgbm_row['best_iteration'])
print('Selected LightGBM parameters:', best_lightgbm_params)
print('Selected LightGBM boosting rounds:', best_lightgbm_iterations)

In [ ]:
xgboost_candidates = [
    {'max_depth': 4, 'min_child_weight': 1, 'gamma': 0.0, 'reg_lambda': 1.0},
    {'max_depth': 6, 'min_child_weight': 1, 'gamma': 0.0, 'reg_lambda': 1.0},
    {'max_depth': 6, 'min_child_weight': 5, 'gamma': 0.1, 'reg_lambda': 5.0},
    {'max_depth': 8, 'min_child_weight': 10, 'gamma': 0.1, 'reg_lambda': 5.0},
]

xgboost_tuning_rows = []
for candidate_number, params in enumerate(xgboost_candidates, start=1):
    started = perf_counter()
    candidate = XGBClassifier(
        n_estimators=500, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9,
        scale_pos_weight=inner_scale_pos_weight,
        objective='binary:logistic', eval_metric='aucpr',
        early_stopping_rounds=40, tree_method='hist', n_jobs=-1,
        random_state=RANDOM_STATE, **params,
    )
    candidate.fit(
        X_inner_fit_encoded, y_inner_fit,
        eval_set=[(X_inner_tune_encoded, y_inner_tune)],
        verbose=False,
    )
    tune_probability = candidate.predict_proba(X_inner_tune_encoded)[:, 1]
    xgboost_tuning_rows.append({
        'candidate': candidate_number,
        'PR-AUC': average_precision_score(y_inner_tune, tune_probability),
        'best_iteration': int(candidate.best_iteration) + 1,
        'fit_seconds': perf_counter() - started,
        **params,
    })
    print(f'XGBoost candidate {candidate_number}/{len(xgboost_candidates)} complete')

xgboost_tuning_results = (
    pd.DataFrame(xgboost_tuning_rows)
      .sort_values('PR-AUC', ascending=False)
      .reset_index(drop=True)
)
display(xgboost_tuning_results.style.format({
    'PR-AUC': '{:.4f}', 'fit_seconds': '{:.1f}',
}))

best_xgboost_row = xgboost_tuning_results.iloc[0]
best_xgboost_params = {
    key: (int(best_xgboost_row[key]) if key in {'max_depth', 'min_child_weight'}
          else float(best_xgboost_row[key]))
    for key in ['max_depth', 'min_child_weight', 'gamma', 'reg_lambda']
}
best_xgboost_iterations = int(best_xgboost_row['best_iteration'])
print('Selected XGBoost parameters:', best_xgboost_params)
print('Selected XGBoost boosting rounds:', best_xgboost_iterations)

In [ ]:
# Refit preprocessing and both winners on the complete earliest-70% development period.
final_preprocessor = clone(preprocessor)
X_train_encoded = final_preprocessor.fit_transform(X_train)
X_validation_encoded = final_preprocessor.transform(X_validation)

tuned_lightgbm = LGBMClassifier(
    n_estimators=best_lightgbm_iterations, learning_rate=0.05,
    class_weight='balanced', subsample=0.9, colsample_bytree=0.9,
    n_jobs=-1, random_state=RANDOM_STATE, verbosity=-1,
    **best_lightgbm_params,
)
tuned_xgboost = XGBClassifier(
    n_estimators=best_xgboost_iterations, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9,
    scale_pos_weight=scale_pos_weight,
    objective='binary:logistic', eval_metric='aucpr',
    tree_method='hist', n_jobs=-1, random_state=RANDOM_STATE,
    **best_xgboost_params,
)

tuned_rows = []
tuned_boosting_models = {
    'Tuned LightGBMClassifier': tuned_lightgbm,
    'Tuned XGBoostClassifier': tuned_xgboost,
}
for name, model in tuned_boosting_models.items():
    started = perf_counter()
    model.fit(X_train_encoded, y_train)
    row = classification_metrics(
        name, model, X_train_encoded, y_train,
        X_validation_encoded, y_validation,
    )
    row['Fit Seconds'] = perf_counter() - started
    tuned_rows.append(row)

tuned_boosting_results = (
    pd.DataFrame(tuned_rows)
      .sort_values(['PR-AUC', 'Recall'], ascending=False)
      .reset_index(drop=True)
)

TUNING_RESULTS_PATH = PROJECT_ROOT / 'Data' / 'processed' / 'boosting_tuning_results.csv'
FINAL_RESULTS_PATH = PROJECT_ROOT / 'Data' / 'processed' / 'tuned_boosting_comparison.csv'
pd.concat([
    lightgbm_tuning_results.assign(model='LightGBM'),
    xgboost_tuning_results.assign(model='XGBoost'),
], ignore_index=True).to_csv(TUNING_RESULTS_PATH, index=False)
tuned_boosting_results.to_csv(FINAL_RESULTS_PATH, index=False)

display(tuned_boosting_results.style.format({
    'Train Accuracy': '{:.4f}', 'Validation Accuracy': '{:.4f}',
    'Precision': '{:.4f}', 'Recall': '{:.4f}', 'F1 Score': '{:.4f}',
    'ROC-AUC': '{:.4f}', 'PR-AUC': '{:.4f}', 'Fit Seconds': '{:.1f}',
}).highlight_max(subset=['F1 Score', 'ROC-AUC', 'PR-AUC'], color='#C6EFCE'))

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for name, model in tuned_boosting_models.items():
    probability = model.predict_proba(X_validation_encoded)[:, 1]
    RocCurveDisplay.from_predictions(y_validation, probability, name=name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_validation, probability, name=name, ax=axes[1])
axes[0].set_title('Tuned Boosting Models: ROC Curves')
axes[1].set_title('Tuned Boosting Models: Precision–Recall Curves')
plt.show()

selected_tuned_model_name = tuned_boosting_results.iloc[0]['Model Name']
selected_tuned_model = tuned_boosting_models[selected_tuned_model_name]
print(f'Selected tuned model by locked validation PR-AUC: {selected_tuned_model_name}')
print(f'Tuning details saved to: {TUNING_RESULTS_PATH}')
print(f'Final comparison saved to: {FINAL_RESULTS_PATH}')

**What this block does:** Refits the selected LightGBM and XGBoost configurations on the complete earliest 70% development period, evaluates them once on the locked latest 30%, saves the detailed results, and selects the final tuned model by PR-AUC.